# Python für Verfahrenstechniker — Teil 3
## NumPy und grafische Darstellung

Bis hierhin haben Sie einzelne Zahlen und Listen verarbeitet. In der Praxis rechnen Sie aber mit
ganzen Kurven: Dampfdruck über einen Temperaturbereich, Konzentration über die Zeit, Kennlinie
einer Pumpe. Dafür gibt es **NumPy** — und um das Ergebnis anzusehen, **Matplotlib**.

---

### Lernziele

Nach diesem Notebook können Sie

- NumPy-Arrays erzeugen und den Unterschied zu Listen erklären,
- ganze Wertebereiche in einer Zeile berechnen (Vektorisierung) statt mit Schleifen,
- Arrays filtern und statistisch auswerten,
- ein lineares Gleichungssystem (z. B. eine Massenbilanz) mit `np.linalg.solve` lösen,
- beschriftete Diagramme mit Matplotlib erstellen, auch mit logarithmischer Achse.

**Voraussetzung:** Teile 1 und 2. **Zeitaufwand:** ca. 120 Minuten.

---
## 1. Warum Listen nicht reichen

Eine Liste ist ein Behälter für beliebige Objekte. Sie ist **nicht** dafür gedacht, mathematisch
mit ihr zu rechnen. Sehen Sie sich an, was `*` und `+` mit Listen machen:

In [ ]:
temperaturen = [78.4, 82.1, 91.5]

print(temperaturen * 2)          # hängt die Liste an sich selbst an!
print(temperaturen + [99.8])     # verkettet zwei Listen

Das ist nicht, was Sie als Ingenieur wollen. Um jede Temperatur einzeln um 273,15 zu erhöhen,
bräuchten Sie eine Schleife. Mit einem **NumPy-Array** rechnen Sie stattdessen direkt:

In [ ]:
import numpy as np

T = np.array([78.4, 82.1, 91.5])   # aus einer Liste ein Array machen

print(T + 273.15)                  # elementweise Addition
print(T * 2)                       # elementweise Multiplikation
print(T ** 0.5)                    # elementweise Wurzel

Das nennt man **Vektorisierung**: Eine Operation wird automatisch auf jedes Element angewendet.
Der Code wird kürzer, lesbarer und um Größenordnungen schneller — bei großen Datenmengen ist das
der entscheidende Unterschied.

> `import numpy as np` ist die überall übliche Abkürzung. Schreiben Sie sie genauso, dann versteht
> jeder Ihren Code.

---
## 2. Arrays erzeugen

| Befehl | Ergebnis |
|--------|----------|
| `np.array([1, 2, 3])` | Array aus einer Liste |
| `np.zeros(5)` | fünf Nullen |
| `np.ones(5)` | fünf Einsen |
| `np.linspace(a, b, n)` | `n` Werte von `a` bis `b`, **inklusive beider Grenzen** |
| `np.arange(a, b, h)` | von `a` bis `b` in Schritten `h`, **ohne `b`** |

Für Kurven ist `linspace` fast immer die richtige Wahl, weil Sie die Anzahl der Stützstellen direkt
vorgeben.

In [ ]:
theta = np.linspace(0, 100, 11)     # 11 Werte von 0 bis 100 °C
print(theta)

print(np.zeros(4))
print(np.arange(0, 1.01, 0.25))

Wichtige Eigenschaften eines Arrays:

In [ ]:
print("Werte:      ", theta)
print("Anzahl:     ", theta.size)
print("Form:       ", theta.shape)
print("Datentyp:   ", theta.dtype)

---
## 3. Rechnen mit ganzen Kurven

Jetzt der eigentliche Gewinn: Die Antoine-Gleichung aus Teil 2 lässt sich auf ein ganzes
Temperaturfeld anwenden — ohne Schleife.

In [ ]:
def dampfdruck_antoine(theta, A, B, C):
    """Dampfdruck in Pa. theta in °C, Konstanten für mmHg/°C.

    Funktioniert für einzelne Werte und für ganze NumPy-Arrays.
    """
    p_mmHg = 10 ** (A - B / (C + theta))
    return p_mmHg * 133.322


A_w, B_w, C_w = 8.07131, 1730.63, 233.426    # Wasser

theta = np.linspace(20, 100, 9)              # °C
p_s = dampfdruck_antoine(theta, A_w, B_w, C_w)

for t, p in zip(theta, p_s):
    print(f"{t:6.1f} °C   ->   {p/1000:8.2f} kPa")

Beachten Sie: Die Funktion aus Teil 2 wurde **nicht verändert**. Sie funktioniert für einzelne
Zahlen genauso wie für Arrays, weil alle darin verwendeten Operationen elementweise arbeiten.
Genau deshalb lohnt es sich, Funktionen sauber zu schreiben.

`zip` verbindet zwei Arrays paarweise — praktisch für Tabellenausgaben.

### Vorsicht: math funktioniert nicht mit Arrays

Verwenden Sie in Funktionen, die auch Arrays verarbeiten sollen, **immer** die NumPy-Varianten:
`np.exp`, `np.log`, `np.sqrt`, `np.sin` statt `math.exp` und so weiter.

In [ ]:
print(np.exp(np.array([0.0, 1.0, 2.0])))
print(np.log(np.array([1.0, 10.0, 100.0])))     # natürlicher Logarithmus
print(np.log10(np.array([1.0, 10.0, 100.0])))   # Zehnerlogarithmus

---
## 4. Ausschneiden, Filtern, Auswerten

Slicing funktioniert wie bei Listen. Zusätzlich können Sie mit einer **Bedingung** filtern —
das ist bei Messdaten außerordentlich nützlich.

In [ ]:
p_kPa = p_s / 1000

print("Erste drei Werte:", p_kPa[:3])
print("Letzter Wert:    ", p_kPa[-1])

In [ ]:
# Boolesche Maske: Wo liegt der Dampfdruck über 50 kPa?
maske = p_kPa > 50
print(maske)

print("Zugehörige Temperaturen:", theta[maske])
print("Zugehörige Drücke:      ", p_kPa[maske])

Die Maske ist ein Array aus `True`/`False`. Verwendet man sie als Index, bleiben nur die Elemente
übrig, an denen `True` steht. So filtern Sie zum Beispiel Ausreißer oder einen Gültigkeitsbereich heraus.

### Statistische Auswertung

In [ ]:
messwerte = np.array([21.4, 24.8, 31.2, 38.7, 45.1, 49.9, 52.3, 21.9])

print(f"Mittelwert:         {messwerte.mean():.2f}")
print(f"Standardabweichung: {messwerte.std(ddof=1):.2f}")
print(f"Minimum / Maximum:  {messwerte.min():.1f} / {messwerte.max():.1f}")
print(f"Position des Maximums: {messwerte.argmax()}")
print(f"Summe:              {messwerte.sum():.1f}")

> `ddof=1` liefert die Stichproben-Standardabweichung (Nenner *n*−1), wie sie in der Messtechnik
> verwendet wird. Ohne dieses Argument rechnet NumPy mit *n* im Nenner.

---
## 5. Lineare Gleichungssysteme: Massenbilanz

Eine Aufgabe, die Sie aus der Bilanzierung kennen: Ein Zulauf wird in einem Trennapparat in drei
Ströme aufgeteilt. Bekannt sind der Gesamtzulauf und die Zusammensetzungen; gesucht sind die drei
Teilströme.

**Beispiel:** Ein Zulauf von 100 kg/h mit 40 % A, 35 % B und 25 % C wird in drei Produktströme
$\dot m_1, \dot m_2, \dot m_3$ getrennt, deren Zusammensetzungen bekannt sind:

| Strom | A | B | C |
|-------|---|---|---|
| 1 | 0,90 | 0,08 | 0,02 |
| 2 | 0,15 | 0,75 | 0,10 |
| 3 | 0,05 | 0,20 | 0,75 |

Die drei Komponentenbilanzen ergeben ein lineares Gleichungssystem $\mathbf{A}\,\vec{x} = \vec{b}$:

$$
\begin{pmatrix}
0{,}90 & 0{,}15 & 0{,}05 \\
0{,}08 & 0{,}75 & 0{,}20 \\
0{,}02 & 0{,}10 & 0{,}75
\end{pmatrix}
\begin{pmatrix} \dot m_1 \\ \dot m_2 \\ \dot m_3 \end{pmatrix}
=
\begin{pmatrix} 40 \\ 35 \\ 25 \end{pmatrix}
$$

Von Hand wäre das mühsam. In NumPy ist es eine Zeile.

In [ ]:
# Koeffizientenmatrix: Zeilen = Komponentenbilanzen, Spalten = Ströme
A = np.array([
    [0.90, 0.15, 0.05],     # Komponente A
    [0.08, 0.75, 0.20],     # Komponente B
    [0.02, 0.10, 0.75],     # Komponente C
])

# rechte Seite: Komponentenmassenströme im Zulauf in kg/h
b = np.array([40.0, 35.0, 25.0])

m_punkt = np.linalg.solve(A, b)

for i, m in enumerate(m_punkt, start=1):
    print(f"Strom {i}: {m:6.2f} kg/h")

print(f"\nSumme: {m_punkt.sum():.2f} kg/h  (Zulauf war 100 kg/h)")

Die Summe muss den Zulauf ergeben — das ist die **Gesamtmassenbilanz** und damit eine unabhängige
Kontrolle Ihrer Rechnung. Führen Sie eine solche Probe grundsätzlich durch.

Prüfen können Sie das Ergebnis auch, indem Sie es wieder einsetzen:

In [ ]:
print("A @ x =", A @ m_punkt)     # @ ist die Matrixmultiplikation
print("b     =", b)
print("maximale Abweichung:", np.max(np.abs(A @ m_punkt - b)))

> Warum die Abweichung nicht exakt null ist und wann `np.linalg.solve` versagt, ist Thema der
> Numerik-Veranstaltung (Gleitkommaarithmetik, Kondition einer Matrix). Für uns genügt: Die
> Restabweichung liegt hier bei etwa 10⁻¹⁴ und ist damit reine Rundung.

---
## 6. Diagramme mit Matplotlib

Ein Diagramm ist in der Verfahrenstechnik oft das eigentliche Ergebnis. Die Bibliothek dafür heißt
**Matplotlib**, das Untermodul `pyplot` wird als `plt` importiert.

In [ ]:
import matplotlib.pyplot as plt

theta = np.linspace(0, 100, 200)                       # feines Temperaturraster
p_s = dampfdruck_antoine(theta, A_w, B_w, C_w) / 1000  # kPa

plt.figure(figsize=(7, 4.5))
plt.plot(theta, p_s)
plt.xlabel("Temperatur $\\vartheta$ / °C")
plt.ylabel("Dampfdruck $p^s$ / kPa")
plt.title("Dampfdruckkurve von Wasser")
plt.grid(True)
plt.show()

### Pflichtbestandteile eines technischen Diagramms

Ein Diagramm ohne Achsenbeschriftung ist wertlos. Merken Sie sich diese Liste — sie gilt für jede
Abbildung in Praktikumsprotokollen, Studien- und Abschlussarbeiten:

1. **Achsenbeschriftung mit Größe und Einheit** (`plt.xlabel`, `plt.ylabel`)
2. **Legende**, sobald mehr als eine Kurve im Bild ist (`plt.legend`)
3. **sinnvoller Achsenbereich** — nicht immer ist der Nullpunkt nötig
4. **Gitternetz** zum Ablesen (`plt.grid(True)`)
5. **Messpunkte als Symbole**, berechnete Kurven als Linie

Die Schreibweise `$\vartheta$` erzeugt LaTeX-Formelsatz in der Beschriftung — praktisch für
griechische Buchstaben und Indizes.

### Mehrere Kurven und Legende

In [ ]:
# Antoine-Konstanten (p in mmHg, theta in °C)
konstanten = {
    "Wasser":  (8.07131, 1730.63, 233.426),
    "Ethanol": (8.20417, 1642.89, 230.300),
    "Benzol":  (6.90565, 1211.03, 220.790),
}

theta = np.linspace(10, 100, 200)

plt.figure(figsize=(7, 4.5))

for stoff, (A_k, B_k, C_k) in konstanten.items():
    p = dampfdruck_antoine(theta, A_k, B_k, C_k) / 1000
    plt.plot(theta, p, label=stoff)

plt.axhline(101.325, color="gray", linestyle="--", linewidth=1, label="Normdruck")

plt.xlabel("Temperatur $\\vartheta$ / °C")
plt.ylabel("Dampfdruck $p^s$ / kPa")
plt.title("Dampfdruckkurven im Vergleich")
plt.legend()
plt.grid(True)
plt.ylim(0, 250)
plt.show()

Die Schnittpunkte der Kurven mit der gestrichelten Linie sind die Normalsiedetemperaturen der
drei Stoffe. Ethanol siedet bei etwa 78 °C, Benzol bei etwa 80 °C, Wasser bei 100 °C — das können
Sie direkt ablesen und mit Tabellenwerten vergleichen.

### Logarithmische Darstellung

Der Dampfdruck folgt näherungsweise $\ln p^s \sim -1/T$. Trägt man $\ln p^s$ über $1/T$ auf,
werden aus den gekrümmten Kurven nahezu Geraden. Diese Auftragung heißt **Clausius-Clapeyron-Diagramm**
und ist der Standard zur Prüfung von Dampfdruckdaten.

In [ ]:
T = np.linspace(283.15, 373.15, 200)      # K
theta = T - 273.15

plt.figure(figsize=(7, 4.5))

for stoff, (A_k, B_k, C_k) in konstanten.items():
    p = dampfdruck_antoine(theta, A_k, B_k, C_k)
    plt.semilogy(1000 / T, p / 1000, label=stoff)

plt.xlabel("$1000\\,/\\,T$ / K$^{-1}$")
plt.ylabel("Dampfdruck $p^s$ / kPa")
plt.title("Dampfdruck in Clausius-Clapeyron-Auftragung")
plt.legend()
plt.grid(True, which="both", linewidth=0.5)
plt.show()

### Mehrere Teilbilder

Mit `plt.subplots` legen Sie mehrere Achsen nebeneinander. Dabei wechselt die Schreibweise leicht:
statt `plt.xlabel(...)` heißt es `ax.set_xlabel(...)`.

In [ ]:
theta = np.linspace(10, 100, 200)
p_wasser = dampfdruck_antoine(theta, *konstanten["Wasser"]) / 1000
p_ethanol = dampfdruck_antoine(theta, *konstanten["Ethanol"]) / 1000

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.5))

ax1.plot(theta, p_wasser, color="tab:blue")
ax1.set_xlabel("$\\vartheta$ / °C")
ax1.set_ylabel("$p^s$ / kPa")
ax1.set_title("Wasser")
ax1.grid(True)

ax2.plot(theta, p_ethanol, color="tab:red")
ax2.set_xlabel("$\\vartheta$ / °C")
ax2.set_ylabel("$p^s$ / kPa")
ax2.set_title("Ethanol")
ax2.grid(True)

fig.tight_layout()
plt.show()

Der Stern in `dampfdruck_antoine(theta, *konstanten["Wasser"])` packt das Tupel `(A, B, C)`
automatisch in die drei einzelnen Parameter aus — eine praktische Abkürzung.

### Messpunkte und Modellkurve zusammen

Die häufigste Abbildung in einem Protokoll: gemessene Punkte, dazu die berechnete Kurve.

In [ ]:
# fiktive Messwerte
theta_mess = np.array([20.0, 40.0, 60.0, 80.0, 95.0])
p_mess     = np.array([2.4, 7.6, 20.1, 47.9, 84.2])     # kPa

theta_fein = np.linspace(15, 100, 200)
p_modell = dampfdruck_antoine(theta_fein, *konstanten["Wasser"]) / 1000

plt.figure(figsize=(7, 4.5))
plt.plot(theta_fein, p_modell, "-", label="Antoine-Gleichung")
plt.plot(theta_mess, p_mess, "o", markersize=7, label="Messwerte")

plt.xlabel("Temperatur $\\vartheta$ / °C")
plt.ylabel("Dampfdruck $p^s$ / kPa")
plt.legend()
plt.grid(True)
plt.show()

Die wichtigsten Formatkürzel als drittes Argument von `plot`:

| Kürzel | Bedeutung |
|--------|-----------|
| `"-"` | durchgezogene Linie |
| `"--"` | gestrichelt |
| `":"` | gepunktet |
| `"o"` | Kreise ohne Linie |
| `"s"` | Quadrate |
| `"o--"` | Kreise mit gestrichelter Linie |

**Konvention:** Messwerte als Symbole ohne verbindende Linie, Modelle als Linie ohne Symbole.
Eine Linie zwischen Messpunkten behauptet, Sie wüssten etwas über den Verlauf dazwischen.

---
## 7. Übungen

### Übung 3.1 — Dichte über der Temperatur

Berechnen Sie mit NumPy die Dichte von Luft (M = 28,96 g/mol) bei p = 1 bar für Temperaturen von
250 K bis 500 K in 26 Schritten. Geben Sie Minimum und Maximum aus und stellen Sie ρ über T
grafisch dar (beschriftet!).

In [ ]:
# Ihre Lösung:

<details>
<summary><b>Lösung anzeigen</b></summary>

```python
T = np.linspace(250, 500, 26)     # K
p = 1e5                            # Pa
M = 28.96e-3                       # kg/mol
R = 8.314                          # J/(mol K)

rho = p * M / (R * T)              # kg/m^3

print(f"Dichte zwischen {rho.min():.3f} und {rho.max():.3f} kg/m^3")

plt.figure(figsize=(7, 4.5))
plt.plot(T, rho)
plt.xlabel("Temperatur $T$ / K")
plt.ylabel(r"Dichte $\rho$ / kg m$^{-3}$")
plt.title("Dichte von Luft bei 1 bar")
plt.grid(True)
plt.show()
```

</details>

### Übung 3.2 — Messdaten filtern

Gegeben sei die folgende Messreihe eines Durchflusssensors in m³/h. Werte über 50 sind
Sensorfehler und sollen verworfen werden.

```python
V_punkt = np.array([12.1, 11.8, 99.9, 12.4, 12.0, 87.3, 11.9, 12.2, 12.5])
```

1. Erzeugen Sie eine Maske für die gültigen Werte.
2. Wie viele Werte bleiben übrig?
3. Berechnen Sie Mittelwert und Standardabweichung der gültigen Werte.

In [ ]:
# Ihre Lösung:
V_punkt = np.array([12.1, 11.8, 99.9, 12.4, 12.0, 87.3, 11.9, 12.2, 12.5])

<details>
<summary><b>Lösung anzeigen</b></summary>

```python
V_punkt = np.array([12.1, 11.8, 99.9, 12.4, 12.0, 87.3, 11.9, 12.2, 12.5])

gueltig = V_punkt < 50
V_ok = V_punkt[gueltig]

print(f"{V_ok.size} von {V_punkt.size} Werten sind gültig.")
print(f"Mittelwert:         {V_ok.mean():.2f} m^3/h")
print(f"Standardabweichung: {V_ok.std(ddof=1):.2f} m^3/h")
```

</details>

### Übung 3.3 — Mischer bilanzieren

Drei Ströme werden zu einem Produktstrom gemischt. Der Produktstrom beträgt 200 kg/h mit den
Massenanteilen 0,50 A / 0,30 B / 0,20 C. Die Zuläufe haben die Zusammensetzungen

| Zulauf | A | B | C |
|--------|---|---|---|
| 1 | 0,80 | 0,15 | 0,05 |
| 2 | 0,20 | 0,70 | 0,10 |
| 3 | 0,10 | 0,10 | 0,80 |

Bestimmen Sie die drei Zulaufmassenströme mit `np.linalg.solve` und prüfen Sie die
Gesamtmassenbilanz.

In [ ]:
# Ihre Lösung:

<details>
<summary><b>Lösung anzeigen</b></summary>

```python
A = np.array([
    [0.80, 0.20, 0.10],
    [0.15, 0.70, 0.10],
    [0.05, 0.10, 0.80],
])

b = np.array([0.50, 0.30, 0.20]) * 200.0     # kg/h je Komponente

m_punkt = np.linalg.solve(A, b)

for i, m in enumerate(m_punkt, start=1):
    print(f"Zulauf {i}: {m:6.2f} kg/h")

print(f"Summe: {m_punkt.sum():.2f} kg/h  (Produkt: 200 kg/h)")
```

</details>

### Übung 3.4 — Pumpen- und Anlagenkennlinie

Die Kennlinie einer Kreiselpumpe lautet näherungsweise

$$H_P(\dot V) = 45 - 0{,}015\,\dot V^2$$

und die Anlagenkennlinie

$$H_A(\dot V) = 12 + 0{,}008\,\dot V^2$$

mit $\dot V$ in m³/h und $H$ in m.

1. Berechnen Sie beide Kurven für $\dot V$ von 0 bis 50 m³/h.
2. Stellen Sie sie in einem Diagramm dar (beschriftet, mit Legende).
3. Bestimmen Sie den Betriebspunkt näherungsweise, indem Sie den Index der kleinsten Differenz
   suchen (`np.argmin` und `np.abs`).

In [ ]:
# Ihre Lösung:

<details>
<summary><b>Lösung anzeigen</b></summary>

```python
V = np.linspace(0, 50, 501)          # m^3/h
H_P = 45 - 0.015 * V ** 2            # m
H_A = 12 + 0.008 * V ** 2            # m

# Betriebspunkt: dort, wo die Differenz minimal wird
i = np.argmin(np.abs(H_P - H_A))
print(f"Betriebspunkt: V = {V[i]:.2f} m^3/h,  H = {H_P[i]:.2f} m")

plt.figure(figsize=(7, 4.5))
plt.plot(V, H_P, label="Pumpenkennlinie")
plt.plot(V, H_A, label="Anlagenkennlinie")
plt.plot(V[i], H_P[i], "ko", markersize=8, label="Betriebspunkt")

plt.xlabel(r"Volumenstrom $\dot V$ / m$^3$ h$^{-1}$")
plt.ylabel("Förderhöhe $H$ / m")
plt.legend()
plt.grid(True)
plt.show()

# Zum Vergleich die analytische Lösung:
V_exakt = np.sqrt((45 - 12) / (0.015 + 0.008))
print(f"analytisch: V = {V_exakt:.2f} m^3/h")
```

</details>

> In Übung 3.4 haben Sie einen Schnittpunkt gesucht, indem Sie das Feld fein abgetastet haben.
> Das funktioniert, ist aber ungenau und teuer. Wie man Nullstellen systematisch und schnell findet
> (Bisektion, Newton-Verfahren), lernen Sie in der Numerik.

---
## Zusammenfassung

| Aufgabe | Befehl |
|---------|--------|
| Import | `import numpy as np`, `import matplotlib.pyplot as plt` |
| Array aus Liste | `np.array([1.0, 2.0])` |
| gleichmäßiges Raster | `np.linspace(0, 100, 201)` |
| elementweise Rechnung | `T + 273.15`, `np.exp(x)` |
| Filtern | `x[x > 50]` |
| Statistik | `x.mean()`, `x.std(ddof=1)`, `x.min()`, `x.argmax()` |
| lineares System | `np.linalg.solve(A, b)` |
| Kurve zeichnen | `plt.plot(x, y, label="...")` |
| Beschriftung | `plt.xlabel`, `plt.ylabel`, `plt.legend`, `plt.grid(True)` |
| log-Achse | `plt.semilogy(x, y)` |

**Merksätze:**

- In Funktionen, die Arrays verarbeiten sollen, `np.exp` statt `math.exp` verwenden.
- Kein Diagramm ohne Achsenbeschriftung mit Einheit.
- Messwerte als Symbole, Modelle als Linie.
- Jede Bilanz mit einer unabhängigen Summenprobe kontrollieren.

---

**Weiter mit Teil 4:** Messdaten einlesen, gute Praxis und das Abschlussprojekt Batch-Reaktor.